# EPIC Template Notebook

This notebook walks through the complete participant workflow for an EPIC forecasting contest. Every contest has three phases:

| Phase | When | What you do |
|-------|------|-------------|
| **Observation** | `start_date` → `end_of_observation` | Watch the live sensor stream; collect data; train your model. |
| **Evaluation** | `end_of_observation` → `end_of_observation + prediction_horizon_seconds` | Stream closes. Ground truth is recorded — hidden from you. |
| **Submission** | Evaluation ends → `end_date` | Submit your forecast for the evaluation window. |

Your submission must contain exactly **`eval_steps` predicted values** for each target variable selected by the organizer.
`eval_steps = round(prediction_horizon_seconds × sampling_rate_hz)` and `target_variables` are shown in the contest's task configuration.

Replace the placeholder values below with the contest and account details from your instructor.

## Installation

This cell installs the EPIC Participant SDK, which you can use to connect to the live sensor stream and submit forecasts. Run it once to set up your environment.

In [ ]:
%pip install "epic-elios-client[notebook]"

## Configure the Client

The following cell create an EPIC client instance and logs in with your credentials. You can reuse the `client` object to interact with the platform throughout the contest.

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"
client = EPICClient(SERVER_URL)

client.login("student1", "correct-password")

##  Browse Available Contests

Look for contests with status `ACTIVE`. The table shows the observation window, evaluation horizon, target variables, and number of steps you must forecast.

In [ ]:
import pandas as pd

contests = client.list_contests(status="ACTIVE")

rows = []
for c in contests:
    task_cfg = c.get("tasks", [{}])[0].get("configuration", {})
    rows.append({
        "contest_id":                c.get("contest_id"),
        "name":                      c.get("name"),
        "twin_id":                   c.get("twin_id"),
        "sampling_rate_hz":          c.get("sampling_rate_hz"),
        "end_of_observation":        c.get("end_of_observation"),
        "prediction_horizon_seconds": c.get("prediction_horizon_seconds"),
        "eval_steps":                task_cfg.get("eval_steps"),
        "target_variables":          ", ".join(task_cfg.get("target_variables", [])),
    })

pd.DataFrame(rows)

Copy a `contest_id` from the table above and paste it here.

In [ ]:
CONTEST_ID = "replace-with-contest-id"  # copy a contest_id from the table above

# Read task configuration from the contest.
contest = next(c for c in contests if c["contest_id"] == CONTEST_ID)
task_config = contest["tasks"][0]["configuration"]
EVAL_STEPS = task_config["eval_steps"]
TARGET_VARIABLES = task_config.get("target_variables") or []
SAMPLING_RATE_HZ = contest["sampling_rate_hz"]

print(f"eval_steps       : {EVAL_STEPS}")
print(f"target_variables : {TARGET_VARIABLES}")
print(f"sampling_rate_hz : {SAMPLING_RATE_HZ} Hz")
print(f"evaluation window: {EVAL_STEPS / SAMPLING_RATE_HZ:.1f} s")

## 3. Register for the Contest

Registration is required before you can join the stream or submit predictions.
Calling `register()` is safe to repeat — if you are already registered the SDK returns a confirmation.

In [ ]:
client.register(CONTEST_ID)

## 4. Collect Live Observations — Observation Phase

`collect()` listens to the live sensor stream and returns a list of observations.
Each observation contains a `sequence_id`, a `timestamp`, and a `sensors` dictionary.

Set `duration_seconds` to cover most of the observation window so you have
enough data to build a good model. The stream closes automatically when the
observation phase ends.

In [ ]:
observations = await client.collect(
    CONTEST_ID,
    duration_seconds=30,          # adjust to match your contest's observation window
    csv_path="epic_observations.csv",
)
print(f"Collected {len(observations)} observations")
observations[:2]

## 5. Inspect the Dataset

Flatten the `sensors` dictionary into columns for easy analysis.

In [ ]:
rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp":   obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df.head()

## 6. Plot a Sensor Channel

In [ ]:
import matplotlib.pyplot as plt

sensor_columns = [c for c in df.columns if c not in {"sequence_id", "timestamp"}]
sensor_name = sensor_columns[0]

df.plot(x="sequence_id", y=sensor_name, figsize=(10, 4), title=f"{sensor_name} over time")
plt.xlabel("Sequence ID")
plt.ylabel(sensor_name)
plt.tight_layout()
plt.show()

## 7. Build a Trivial Prediction — Persistence Baseline

For a first submission, use the simplest possible baseline: predict that every
future value stays equal to the last observed value.
This is not competitive but verifies the submission pipeline.

Your forecast must contain **exactly `eval_steps` values** for each target variable selected by the organizer.

In [ ]:
forecast_targets = TARGET_VARIABLES or sensor_columns
missing_targets = [target for target in forecast_targets if target not in df.columns]
if missing_targets:
    raise ValueError(f"Target variables not present in collected data: {missing_targets}")

last_values = df[forecast_targets].iloc[-1]

# Persist the last observed value for every target variable and every step.
payload = {
    "forecast": {
        target: [float(last_values[target])] * EVAL_STEPS
        for target in forecast_targets
    }
}

print(f"Forecast shape: {len(forecast_targets)} target variable(s) × {EVAL_STEPS} steps")
payload

## 8. Submit the Prediction — Submission Window

Submissions are only accepted **after the evaluation window closes**
(`end_of_observation + prediction_horizon_seconds`).
If you try to submit too early, the server returns a 409 error with a message
telling you when the submission window opens.

In [ ]:
submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    payload=payload,
)
submission

## 9. Check Scores and Leaderboard

Scoring runs asynchronously — a new submission may stay `PENDING` briefly.
Re-run this cell until the status changes to `EVALUATED`.

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

print("=== My Submissions ===")
for sub in scores.get("submissions", []):
    print(f"  {sub['submission_id'][:8]}…  status={sub['status']}  "
          f"scores={[round(s['value'], 4) for s in sub.get('scores', [])]}")

print("\n=== Leaderboard ===")
for entry in leaderboard.get("entries", []):
    print(f"  Rank {entry['rank']}  user={entry['user_id'][:8]}…  score={entry['score']:.4f}")

## Next Steps

The persistence baseline is a starting point, not a winning strategy.
To improve your score:

- **Fit a physics model** — see `mass_spring_damper_forecasting.ipynb` for an example using parameter estimation and numerical integration.
- **Train a data-driven model** — fit an AR(p), LSTM, or Transformer on the collected observations.
- **Use the full observation window** — the more data you collect, the better your parameter estimates or training set.